In [9]:
import pandas as pd
import numpy as np

In [60]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
apartment_for_rent_classified = fetch_ucirepo(id=555) 
  
# data (as pandas dataframes) 
df = apartment_for_rent_classified.data.features 
  
# metadata 
print(apartment_for_rent_classified.metadata) 
  
# variable information 
print(apartment_for_rent_classified.variables) 


{'uci_id': 555, 'name': 'Apartment for Rent Classified', 'repository_url': 'https://archive.ics.uci.edu/dataset/555/apartment+for+rent+classified', 'data_url': 'https://archive.ics.uci.edu/static/public/555/data.csv', 'abstract': 'This is a dataset of classified for apartments for rent in USA.\n', 'area': 'Business', 'tasks': ['Classification', 'Regression', 'Clustering'], 'characteristics': ['Multivariate'], 'num_instances': 10000, 'num_features': 21, 'feature_types': ['Categorical', 'Integer'], 'demographics': [], 'target_col': None, 'index_col': ['id'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2019, 'last_updated': 'Mon Feb 26 2024', 'dataset_doi': '10.24432/C5X623', 'creators': [], 'intro_paper': None, 'additional_info': {'summary': "The dataset contains of 10'000 or 100'000 rows and of 22 columns The data has been cleaned in the way that \r\ncolumn price and square_feet never is empty but the dataset is saved as it was created.\r\n\r\n

/opt/anaconda3/envs/pydata/lib/python3.11/site-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (0,5,6,12,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


### Notes/Ideas, Process?

- Create repo, do the whole GIT thing
- Data exploration?
    - Summary stats, for price primarily
    - Visuals, general graphs, histograms, pairplots, etc.
    - Correlations, relationships
    - Basic linear models?
- Strategy for training, testing
    - Initially split into train and test datasets (80/20? 75/25?)
    - Run CV on training data
    - Evaluate models on test data
- Establish Evaluation Strategy
    - RMSE
    - R-squared
    - Mean Absolute Error?
    - Residual analysis?
    - Sensitivity analysis?
- Establish CV, hyperparameter tuning Strategy
    - Custom function for gridsearchCV?? or Random search CV?
    - Maybe different approaches for different model groups
- Simpler experimental ML models - regression based
    - Decision Trees
    - KNN - SCALING
    - SVR ??? - SCALING
    - Regularized regression - SCALING
- Ensemble methods
    - Random Forest
    - XGBoost
    - Custom ensemble
- Model Evaluation and comparison
    - Tables, visuals


- Make two helper functions?
    - One pipeline for training and scaling,
    - One pipeline for CV, tuning, and evaluation


### Crucial Best Practices for This Workflow

To make sure this workflow is mathematically sound, keep these three execution rules in mind:

- **Fit Transformers ONLY on the Training Data**: If you are scaling features (e.g., StandardScaler), computing the mean, or imputing missing values, you must .fit() your scaler only on the training data. You then .transform() both the training and test data. Fitting a scaler on the full dataset before splitting leaks the global mean and variance into your test set.

- **Keep Random Seeds Consistent**: When comparing multiple models using CV on the training set, ensure you lock the random state or use the exact same cross-validation splits (e.g., passing the same KFold object to all models). This guarantees that Model A and Model B are being evaluated on the exact same subsets of data.

- **Never Go Backward**: If your final test set score is disappointing, do not go back and tweak your hyperparameters to make the test score better. If you do, your test set is no longer a true test set; it has become part of your tuning process, and your final metrics will be overly optimistic.

# To Do: 

- Make new variables from amenities, pets_allowed
- Create dummies for state?
- Pull out month from time
- Can I use top cities? Create dummies from them?

In [25]:
df.head()

,category,title,body,amenities,bathrooms,bedrooms,currency,fee,has_photo,pets_allowed,...,price_display,price_type,square_feet,address,cityname,state,latitude,longitude,source,time
0,housing/rent/apartment,One BR 507 & 509 Esplanade,"This unit is located at 507 & 509 Esplanade, R...",NaN,1,1,USD,No,Thumbnail,Cats,...,2195,Monthly,542,507 509 Esplanade,Redondo Beach,CA,33.8520,-118.3759,RentLingo,1.577360e+09
1,housing/rent/apartment,Three BR 146 Lochview Drive,"This unit is located at 146 Lochview Drive, Ne...",NaN,1.5,3,USD,No,Thumbnail,"Cats,Dogs",...,1250,Monthly,1500,146 Lochview Dr,Newport News,VA,37.0867,-76.4941,RentLingo,1.577360e+09
2,housing/rent/apartment,Three BR 3101 Morningside Drive,This unit is located at 3101 Morningside Drive...,NaN,2,3,USD,No,Thumbnail,NaN,...,1395,Monthly,1650,3101 Morningside Dr,Raleigh,NC,35.8230,-78.6438,RentLingo,1.577360e+09
3,housing/rent/apartment,Two BR 209 Aegean Way,"This unit is located at 209 Aegean Way, Vacavi...",NaN,1,2,USD,No,Thumbnail,"Cats,Dogs",...,1600,Monthly,820,209 Aegean Way,Vacaville,CA,38.3622,-121.9712,RentLingo,1.577360e+09
4,housing/rent/apartment,One BR 4805 Marquette NE,"This unit is located at 4805 Marquette NE, Alb...",NaN,1,1,USD,No,Thumbnail,"Cats,Dogs",...,975,Monthly,624,4805 Marquette NE,Albuquerque,NM,35.1038,-106.6110,RentLingo,1.577360e+09


In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99826 entries, 0 to 99825
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   category       99826 non-null  object 
 1   title          99826 non-null  object 
 2   body           99826 non-null  object 
 3   amenities      83749 non-null  object 
 4   bathrooms      99760 non-null  object 
 5   bedrooms       99699 non-null  object 
 6   currency       99822 non-null  object 
 7   fee            99823 non-null  object 
 8   has_photo      99823 non-null  object 
 9   pets_allowed   39192 non-null  object 
 10  price          99821 non-null  float64
 11  price_display  99820 non-null  object 
 12  price_type     99823 non-null  object 
 13  square_feet    99823 non-null  object 
 14  address        7946 non-null   object 
 15  cityname       99521 non-null  object 
 16  state          99521 non-null  object 
 17  latitude       99797 non-null  float64
 18  longit

In [27]:
df.isna().sum()

category             0
title                0
body                 0
amenities        16077
bathrooms           66
bedrooms           127
currency             4
fee                  3
has_photo            3
pets_allowed     60634
price                5
price_display        6
price_type           3
square_feet          3
address          91880
cityname           305
state              305
latitude            29
longitude           31
source               6
time                 6
dtype: int64

Drop Null values applied to columns with small numbers of Null values.

In [61]:
cols_to_drop = df.columns[df.isna().sum() <= 500]

df = df.dropna(subset=cols_to_drop)

In [ ]:
# Large amounts of null values in both pets_allowed, and address
# Likely just remove both
# Can I produde other variables from amenities? e.g. 'has_ac', 'has_pool', etc.

df.isna().sum()

category             0
title                0
body                 0
amenities        15871
bathrooms            0
bedrooms             0
currency             0
fee                  0
has_photo            0
pets_allowed     60321
price                0
price_display        0
price_type           0
square_feet          0
address          91508
cityname             0
state                0
latitude             0
longitude            0
source               0
time                 0
dtype: int64

In [50]:
print(df['amenities'].value_counts())
print(df['pets_allowed'].value_counts())

amenities
Parking                                                                                                                    6170
Parking,Storage                                                                                                            2116
Gym,Pool                                                                                                                   1874
Pool                                                                                                                       1480
Gym,Parking,Pool                                                                                                           1190
                                                                                                                           ... 
AC,Cable or Satellite,Dishwasher,Fireplace,Garbage Disposal,Patio/Deck,Refrigerator,Tennis,Washer Dryer                       1
Gated,Parking,Pool,Washer Dryer,Wood Floors                                                   

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99335 entries, 0 to 99825
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   category       99335 non-null  object 
 1   title          99335 non-null  object 
 2   body           99335 non-null  object 
 3   amenities      83464 non-null  object 
 4   bathrooms      99335 non-null  object 
 5   bedrooms       99335 non-null  object 
 6   currency       99335 non-null  object 
 7   fee            99335 non-null  object 
 8   has_photo      99335 non-null  object 
 9   pets_allowed   39014 non-null  object 
 10  price          99335 non-null  float64
 11  price_display  99335 non-null  object 
 12  price_type     99335 non-null  object 
 13  square_feet    99335 non-null  object 
 14  address        7827 non-null   object 
 15  cityname       99335 non-null  object 
 16  state          99335 non-null  object 
 17  latitude       99335 non-null  float64
 18  longitude  

In [ ]:
df['price'].describe()

count    99819.000000
mean      1527.224015
std        903.638141
min        100.000000
25%       1014.000000
50%       1350.000000
75%       1795.000000
max      52500.000000
Name: price, dtype: float64

In [52]:
# Is there any difference between price, price_display, and currency? 
# Currency not needed, USD for all.
# price_display same as price, need to remove one to avoid leakage.

df[['price', 'currency', 'price_display']].head()

,price,currency,price_display
0,2195.0,USD,2195
1,1250.0,USD,1250
2,1395.0,USD,1395
3,1600.0,USD,1600
4,975.0,USD,975


In [63]:
# Weekly vs monthly will make the amounts very different, remove those with type Weekly.

df['price_type'].value_counts()

price_type
Monthly    99332
Name: count, dtype: int64

In [62]:
df = df[df['price_type'] == 'Monthly']

In [64]:
df_reduced = df.drop(
    columns=['price_display', 'price_type', 'currency', 'address', 'category']
    )

df_reduced.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99332 entries, 0 to 99825
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         99332 non-null  object 
 1   body          99332 non-null  object 
 2   amenities     83461 non-null  object 
 3   bathrooms     99332 non-null  object 
 4   bedrooms      99332 non-null  object 
 5   fee           99332 non-null  object 
 6   has_photo     99332 non-null  object 
 7   pets_allowed  39013 non-null  object 
 8   price         99332 non-null  float64
 9   square_feet   99332 non-null  object 
 10  cityname      99332 non-null  object 
 11  state         99332 non-null  object 
 12  latitude      99332 non-null  float64
 13  longitude     99332 non-null  float64
 14  source        99332 non-null  object 
 15  time          99332 non-null  float64
dtypes: float64(4), object(12)
memory usage: 12.9+ MB
